In [1]:
import mysql.connector
import sqlite3
from mysql.connector import Error

In [2]:
import mysql.connector
import sqlite3
from mysql.connector import Error
from decimal import Decimal
import datetime
import os
import random

def convert_decimal_to_float(obj):
    """
    Рекурсивно преобразует объекты Decimal в float.
    Также преобразует datetime в строки для SQLite.
    """
    if isinstance(obj, Decimal):
        return float(obj)
    elif isinstance(obj, (datetime.datetime, datetime.date)):
        return str(obj)
    elif isinstance(obj, tuple):
        return tuple(convert_decimal_to_float(item) for item in obj)
    elif isinstance(obj, list):
        return [convert_decimal_to_float(item) for item in obj]
    else:
        return obj

def create_local_db_and_copy_data(db_directory, db_filename='local_credit_database.db', num_rows=40):
    """
    Создает локальную базу данных SQLite и копирует в нее случайные строки из удаленной MariaDB.

    :param db_directory: директория, в которой будет храниться локальная база данных
    :param db_filename: имя файла локальной базы данных
    :param num_rows: количество случайных строк для выборки из каждой таблицы
    """
    # Создаем директорию, если она не существует
    os.makedirs(db_directory, exist_ok=True)
    
    # Полный путь к файлу базы данных
    local_db_file = os.path.join(db_directory, db_filename)

    # Удаляем файл базы данных, если он уже существует
    if os.path.exists(local_db_file):
        os.remove(local_db_file)
        print(f"Существующий файл базы данных {local_db_file} удален")

    remote_host = 'relational.fel.cvut.cz'
    remote_port = 3306
    remote_user = 'guest'
    remote_password = 'ctu-relational'
    remote_database = 'Credit'

    # Подключение к удаленной базе данных
    try:
        remote_conn = mysql.connector.connect(
            host=remote_host,
            port=remote_port,
            user=remote_user,
            password=remote_password,
            database=remote_database
        )
        print("Успешное подключение к удаленной базе данных")
    except Error as e:
        print(f"Ошибка подключения к удаленной базе данных: {e}")
        return

    # Подключение к локальной SQLite базе данных
    try:
        local_conn = sqlite3.connect(local_db_file)
        print(f"Создана локальная база данных SQLite: {local_db_file}")
    except Error as e:
        print(f"Ошибка подключения к локальной базе данных: {e}")
        remote_conn.close()
        return

    cursor_remote = remote_conn.cursor()
    cursor_local = local_conn.cursor()

    # Определяем имена таблиц
    table_names = [
        'category', 'charge', 'corporation', 'member', 'payment', 'provider', 'region', 'statement'
    ]

    # Создаем таблицы в локальной базе данных с внешними ключами
    create_table_sql = {
        'region': '''
            CREATE TABLE IF NOT EXISTS region (
                region_no INTEGER PRIMARY KEY,
                region_name TEXT NOT NULL,
                street TEXT NOT NULL,
                city TEXT NOT NULL,
                state_prov TEXT NOT NULL,
                country TEXT NOT NULL,
                mail_code TEXT NOT NULL,
                phone_no TEXT NOT NULL,
                region_code TEXT NOT NULL
            )
        ''',
        'category': '''
            CREATE TABLE IF NOT EXISTS category (
                category_no INTEGER PRIMARY KEY,
                category_desc TEXT NOT NULL,
                category_code TEXT NOT NULL
            )
        ''',
        'corporation': '''
            CREATE TABLE IF NOT EXISTS corporation (
                corp_no INTEGER PRIMARY KEY,
                corp_name TEXT NOT NULL,
                street TEXT NOT NULL,
                city TEXT NOT NULL,
                state_prov TEXT NOT NULL,
                country TEXT NOT NULL,
                mail_code TEXT NOT NULL,
                phone_no TEXT NOT NULL,
                expr_dt TEXT NOT NULL,
                region_no INTEGER NOT NULL,
                corp_code TEXT NOT NULL,
                FOREIGN KEY (region_no) REFERENCES region (region_no) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'provider': '''
            CREATE TABLE IF NOT EXISTS provider (
                provider_no INTEGER PRIMARY KEY,
                provider_name TEXT NOT NULL,
                street TEXT NOT NULL,
                city TEXT NOT NULL,
                state_prov TEXT NOT NULL,
                mail_code TEXT NOT NULL,
                country TEXT NOT NULL,
                phone_no TEXT NOT NULL,
                issue_dt TEXT NOT NULL,
                expr_dt TEXT NOT NULL,
                region_no INTEGER NOT NULL,
                provider_code TEXT NOT NULL,
                FOREIGN KEY (region_no) REFERENCES region (region_no) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'member': '''
            CREATE TABLE IF NOT EXISTS member (
                member_no INTEGER PRIMARY KEY,
                lastname TEXT NOT NULL,
                firstname TEXT NOT NULL,
                middleinitial TEXT,
                street TEXT NOT NULL,
                city TEXT NOT NULL,
                state_prov TEXT NOT NULL,
                country TEXT NOT NULL,
                mail_code TEXT NOT NULL,
                phone_no TEXT,
                photograph BLOB,
                issue_dt TEXT NOT NULL,
                expr_dt TEXT NOT NULL,
                region_no INTEGER NOT NULL,
                corp_no INTEGER,
                prev_balance REAL,
                curr_balance REAL,
                member_code TEXT NOT NULL,
                FOREIGN KEY (region_no) REFERENCES region (region_no) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (corp_no) REFERENCES corporation (corp_no) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'statement': '''
            CREATE TABLE IF NOT EXISTS statement (
                statement_no INTEGER PRIMARY KEY,
                member_no INTEGER NOT NULL,
                statement_dt TEXT NOT NULL,
                due_dt TEXT NOT NULL,
                statement_amt REAL NOT NULL,
                statement_code TEXT NOT NULL,
                FOREIGN KEY (member_no) REFERENCES member (member_no) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'charge': '''
            CREATE TABLE IF NOT EXISTS charge (
                charge_no INTEGER PRIMARY KEY,
                member_no INTEGER NOT NULL,
                provider_no INTEGER NOT NULL,
                category_no INTEGER NOT NULL,
                charge_dt TEXT NOT NULL,
                charge_amt REAL NOT NULL,
                statement_no INTEGER NOT NULL,
                charge_code TEXT NOT NULL,
                FOREIGN KEY (category_no) REFERENCES category (category_no) ON DELETE NO ACTION ON UPDATE NO ACTION,
                FOREIGN KEY (member_no) REFERENCES member (member_no) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (provider_no) REFERENCES provider (provider_no) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (statement_no) REFERENCES statement (statement_no) ON DELETE NO ACTION ON UPDATE NO ACTION
            )
        ''',
        'payment': '''
            CREATE TABLE IF NOT EXISTS payment (
                payment_no INTEGER PRIMARY KEY,
                member_no INTEGER NOT NULL,
                payment_dt TEXT NOT NULL,
                payment_amt REAL NOT NULL,
                statement_no INTEGER,
                payment_code TEXT NOT NULL,
                FOREIGN KEY (member_no) REFERENCES member (member_no) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (statement_no) REFERENCES statement (statement_no) ON DELETE CASCADE ON UPDATE CASCADE
            )
        '''
    }

    # Важно создавать таблицы в правильном порядке, чтобы внешние ключи работали
    creation_order = [
        'region', 'category', 'corporation', 'provider', 'member', 'statement', 'charge', 'payment'
    ]

    for table_name in creation_order:
        cursor_local.execute(create_table_sql[table_name])

    # Отключаем проверку внешних ключей на время вставки данных
    cursor_local.execute("PRAGMA foreign_keys = OFF;")

    # Копируем данные из каждой таблицы (по num_rows случайных строк)
    for table_name in table_names:
        print(f"Копируем данные из таблицы {table_name}...")
        
        # Сначала получаем общее количество строк в таблице
        count_query = f"SELECT COUNT(*) FROM {table_name}"
        cursor_remote.execute(count_query)
        total_rows = cursor_remote.fetchone()[0]
        
        # Если в таблице меньше или столько же строк, сколько нужно выбрать, 
        # просто выбираем все строки
        if total_rows <= num_rows:
            select_query = f"SELECT * FROM {table_name}"
            cursor_remote.execute(select_query)
            rows = cursor_remote.fetchall()
        else:
            # Иначе выбираем случайные строки
            # Получаем все строки
            select_query = f"SELECT * FROM {table_name}"
            cursor_remote.execute(select_query)
            all_rows = cursor_remote.fetchall()
            
            # Выбираем случайные строки
            random_rows = random.sample(all_rows, num_rows)
            rows = random_rows

        # Преобразуем Decimal в float и datetime в строки
        converted_rows = [convert_decimal_to_float(row) for row in rows]

        # Получаем количество столбцов для формирования SQL-запроса
        placeholders = ','.join(['?' for _ in range(len(cursor_remote.description))])
        insert_query = f"INSERT INTO {table_name} VALUES ({placeholders})"

        cursor_local.executemany(insert_query, converted_rows)
        print(f"Скопировано {len(converted_rows)} случайных строк из таблицы {table_name}")

    # Включаем проверку внешних ключей
    cursor_local.execute("PRAGMA foreign_keys = ON;")

    # Сохраняем изменения и закрываем соединения
    local_conn.commit()
    remote_conn.close()
    local_conn.close()
    print(f"Данные успешно скопированы в локальную базу данных: {local_db_file}")

if __name__ == "__main__":
    # Пример использования:
    # Укажите директорию, в которой хотите хранить базу данных
    directory = "."
    create_local_db_and_copy_data(db_directory=directory, num_rows=40)

Существующий файл базы данных ./local_credit_database.db удален
Успешное подключение к удаленной базе данных
Создана локальная база данных SQLite: ./local_credit_database.db
Копируем данные из таблицы category...
Скопировано 10 случайных строк из таблицы category
Копируем данные из таблицы charge...
Скопировано 40 случайных строк из таблицы charge
Копируем данные из таблицы corporation...
Скопировано 40 случайных строк из таблицы corporation
Копируем данные из таблицы member...
Скопировано 40 случайных строк из таблицы member
Копируем данные из таблицы payment...
Скопировано 40 случайных строк из таблицы payment
Копируем данные из таблицы provider...
Скопировано 40 случайных строк из таблицы provider
Копируем данные из таблицы region...
Скопировано 9 случайных строк из таблицы region
Копируем данные из таблицы statement...
Скопировано 40 случайных строк из таблицы statement
Данные успешно скопированы в локальную базу данных: ./local_credit_database.db


In [3]:
import sqlite3
import pandas as pd
from io import StringIO
import csv

def get_database_info_and_csv():
    # Подключаемся к локальной базе данных
    conn = sqlite3.connect('local_credit_database.db')
    cursor = conn.cursor()

    # Получаем список таблиц
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    output = StringIO()
    writer = csv.writer(output)

    for table_name in tables:
        table_name = table_name[0]
        print(f"Обработка таблицы: {table_name}")

        # Получаем структуру таблицы (CREATE TABLE)
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_table_sql = cursor.fetchone()[0]
        output.write(f"\n--- Структура таблицы {table_name} ---\n")
        output.write(f"{create_table_sql}\n")

        # Получаем и выводим все данные из таблицы
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", conn)
        output.write(f"\n--- Данные таблицы {table_name} ---\n")
        
        # Записываем заголовки
        writer.writerow(df.columns.tolist())
        # Записываем данные
        for row in df.itertuples(index=False, name=None):
            writer.writerow(row)
        output.write("\n")  # Добавляем пустую строку между таблицами

    conn.close()
    
    return output.getvalue()

# Вызов функции и вывод результата
result = get_database_info_and_csv()
print(result)

Обработка таблицы: region
Обработка таблицы: category
Обработка таблицы: corporation
Обработка таблицы: provider
Обработка таблицы: member
Обработка таблицы: statement
Обработка таблицы: charge
Обработка таблицы: payment

--- Структура таблицы region ---
CREATE TABLE region (
                region_no INTEGER PRIMARY KEY,
                region_name TEXT NOT NULL,
                street TEXT NOT NULL,
                city TEXT NOT NULL,
                state_prov TEXT NOT NULL,
                country TEXT NOT NULL,
                mail_code TEXT NOT NULL,
                phone_no TEXT NOT NULL,
                region_code TEXT NOT NULL
            )

--- Данные таблицы region ---
region_no,region_name,street,city,state_prov,country,mail_code,phone_no,region_code
1,North American,111 First St.,Toronto,ON,Ca,,,
2,South American,222 Second St.,Sao Paulo,,Br,,,
3,Scandanavian,333 Third St.,Goteborg,,Sw,,,
4,Western Europea,444 Fourth St.,Brussels,,,,,
5,Eastern Europea,555 Fifth St St,Bud

In [4]:
import sqlite3
import pandas as pd

def execute_sql_query(db_file, query):
    """
    Выполняет SQL-запрос к локальной базе данных и возвращает результат в виде DataFrame.

    :param db_file: путь к файлу локальной базы данных SQLite
    :param query: SQL-запрос для выполнения
    :return: результат запроса в виде pandas DataFrame
    """
    # Подключаемся к базе данных
    conn = sqlite3.connect(db_file)
    
    try:
        # Выполняем запрос и возвращаем результат в виде DataFrame
        result_df = pd.read_sql_query(query, conn)
        return result_df
    except Exception as e:
        print(f"Произошла ошибка при выполнении запроса: {e}")
        return None
    finally:
        # Закрываем соединение
        conn.close()

# Пример использования:
# db_file = 'local_credit_database.db'
# query = 'SELECT * FROM member LIMIT 10;'
# result = execute_sql_query(db_file, query)
# print(result)

In [5]:
query = """
SELECT 
    SUM(ch.charge_amt) AS total_meals_charges_from_M_regions
FROM 
    charge ch
JOIN 
    category cat ON ch.category_no = cat.category_no
JOIN 
    provider pr ON ch.provider_no = pr.provider_no
JOIN 
    region r ON pr.region_no = r.region_no
WHERE 
    cat.category_desc = 'Meals'
    AND r.region_name LIKE 'M%';
"""

In [6]:
db_file = 'local_credit_database.db'
result = execute_sql_query(db_file, query)
print(result)

   total_meals_charges_from_M_regions
0                              3892.0
